In [1]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 │ Graph Data Structure + BFS-based Heuristic
# ─────────────────────────────────────────────────────────────────────────────
# Heuristic  h(n) = BFS_hops(n, goal) × min_edge_cost
#   • BFS runs on the unweighted undirected graph from the goal node.
#   • min_edge_cost is the smallest edge weight in the entire graph.
#   • Because every hop costs ≥ min_edge_cost, h(n) ≤ actual remaining cost
#     → the heuristic is ADMISSIBLE and CONSISTENT, guaranteeing optimality.
# ─────────────────────────────────────────────────────────────────────────────

from collections import defaultdict, deque


class Graph:
    """
    Undirected weighted graph for the Smart Waste Collection network.

    Attributes
    ----------
    graph        : adjacency list  {node: [(neighbour, weight), ...]}
    min_edge_cost: minimum edge weight seen across all add_edge() calls
    nodes        : set of all node names present in the graph
    """

    def __init__(self):
        self.graph         = defaultdict(list)
        self.min_edge_cost = float('inf')
        self.nodes         = set()

    # ── Edge insertion ────────────────────────────────────────────────────────
    def add_edge(self, u: str, v: str, weight: float) -> None:
        """
        Add an undirected edge between nodes u and v with the given weight.

        Parameters
        ----------
        u, v   : node names (non-empty strings)
        weight : travel cost / fuel consumption (must be > 0)

        Raises
        ------
        ValueError : if node names are empty or weight is non-positive.
        """
        if not u or not v:
            raise ValueError(
                f"[add_edge] Node names must be non-empty strings. "
                f"Received: u='{u}', v='{v}'"
            )
        if weight <= 0:
            raise ValueError(
                f"[add_edge] Edge weight must be positive. "
                f"Received weight={weight} for edge ({u} ↔ {v})"
            )

        self.graph[u].append((v, weight))
        self.graph[v].append((u, weight))
        self.nodes.add(u)
        self.nodes.add(v)
        self.min_edge_cost = min(self.min_edge_cost, weight)

    # ── Neighbour access ──────────────────────────────────────────────────────
    def get_neighbors(self, node: str) -> list:
        """
        Return neighbours of *node* sorted by edge weight (ascending).
        Lower-cost roads are explored first, which prunes the IDA* search
        tree earlier and reduces total nodes explored.
        """
        if node not in self.nodes:
            print(f"[get_neighbors] Warning: node '{node}' not found in graph.")
            return []
        return sorted(self.graph[node], key=lambda x: x[1])

    # ── BFS heuristic ─────────────────────────────────────────────────────────
    def compute_heuristic(self, goal: str) -> dict:
        """
        Compute h(n) for every node using BFS on the unweighted graph.

        h(n) = hop_count(n → goal) × min_edge_cost

        The hop count is found by a reverse BFS starting at *goal*; on an
        undirected graph this equals the forward hop count.

        Parameters
        ----------
        goal : destination node name

        Returns
        -------
        dict  {node: h_value}   (unreachable nodes → float('inf'))
        """
        if goal not in self.nodes:
            raise ValueError(
                f"[compute_heuristic] Goal node '{goal}' not found in graph."
            )
        if self.min_edge_cost == float('inf'):
            raise ValueError(
                "[compute_heuristic] Graph has no edges; cannot compute heuristic."
            )

        # BFS – count hops from goal to every other node
        hops   = {goal: 0}
        queue  = deque([goal])

        while queue:
            current = queue.popleft()
            for neighbour, _ in self.graph[current]:
                if neighbour not in hops:
                    hops[neighbour] = hops[current] + 1
                    queue.append(neighbour)

        # Build heuristic dict; nodes not reached by BFS are unreachable
        h = {}
        for node in self.nodes:
            if node in hops:
                h[node] = hops[node] * self.min_edge_cost
            else:
                h[node] = float('inf')

        return h


print("Cell 1 loaded: Graph class with BFS heuristic ready.")


Cell 1 loaded: Graph class with BFS heuristic ready.


In [2]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 │ Input File Parser
# ─────────────────────────────────────────────────────────────────────────────
# Reads inputPS1.txt which may contain multiple CASE blocks.
#
# Expected block format (keywords are case-insensitive):
#   CASE   <n>
#   NODES  <n>
#   EDGES  <n>
#   <node1> <node2> <weight>     ← repeated EDGES times
#   HEURISTIC <node> <value>     ← parsed then discarded (heuristic is computed)
#   SOURCE      <node>
#   DESTINATION <node>
#
# Blank lines between blocks are tolerated.
# ─────────────────────────────────────────────────────────────────────────────

import os


def parse_input(filename: str) -> list:
    """
    Parse *filename* and return a list of case dictionaries.

    Each dictionary has the keys:
        case_num    (int)   – 1-based case index
        source      (str)   – source node name
        destination (str)   – destination node name
        graph       (Graph) – populated Graph object

    Raises
    ------
    FileNotFoundError : if *filename* does not exist.
    """
    if not os.path.isfile(filename):
        raise FileNotFoundError(
            f"[parse_input] Input file '{filename}' not found."
        )

    with open(filename, 'r') as fh:
        raw_lines = fh.readlines()

    # Strip whitespace and skip blank lines; keep (original_index, text) for
    # helpful error messages.
    lines = [
        (i + 1, ln.strip())
        for i, ln in enumerate(raw_lines)
        if ln.strip()
    ]

    cases           = []
    i               = 0
    total_lines     = len(lines)

    while i < total_lines:
        lineno, text = lines[i]
        tokens       = text.split()

        # ── Expect CASE keyword to start a new block ─────────────────────────
        if tokens[0].upper() != 'CASE':
            i += 1
            continue

        case_num = int(tokens[1]) if len(tokens) > 1 else len(cases) + 1
        i       += 1

        graph       = Graph()
        source      = None
        destination = None
        edge_count  = 0          # declared number of edges
        edges_read  = 0          # edges actually parsed
        in_edges    = False      # True while reading edge triples

        while i < total_lines:
            lineno, text = lines[i]
            tokens       = text.split()

            # New CASE starts → finish this block without consuming the line
            if tokens[0].upper() == 'CASE':
                break

            keyword = tokens[0].upper()

            if keyword == 'NODES':
                # Number of nodes – informational only (nodes added via edges)
                i += 1

            elif keyword == 'EDGES':
                edge_count = int(tokens[1])
                in_edges   = True
                i         += 1

            elif keyword == 'HEURISTIC':
                # Pre-defined heuristic value – parsed and discarded.
                # Our algorithm recomputes h(n) via BFS × min_edge_cost.
                in_edges = False
                i       += 1

            elif keyword == 'SOURCE':
                in_edges = False
                if len(tokens) < 2:
                    print(f"[parse_input] Line {lineno}: SOURCE keyword missing node name – skipping.")
                else:
                    source = tokens[1]
                i += 1

            elif keyword == 'DESTINATION':
                in_edges = False
                if len(tokens) < 2:
                    print(f"[parse_input] Line {lineno}: DESTINATION keyword missing node name – skipping.")
                else:
                    destination = tokens[1]
                i += 1

            elif in_edges and len(tokens) == 3:
                # Edge definition: <node1> <node2> <weight>
                u, v = tokens[0], tokens[1]
                try:
                    weight = float(tokens[2])
                    graph.add_edge(u, v, weight)
                    edges_read += 1
                except ValueError as exc:
                    print(f"[parse_input] Line {lineno}: invalid edge – {exc}")
                i += 1

            else:
                # Unrecognised line inside a block – skip gracefully
                i += 1

        # ── Validate the completed block ──────────────────────────────────────
        if source is None:
            print(f"[parse_input] Case {case_num}: SOURCE not found – case skipped.")
            continue
        if destination is None:
            print(f"[parse_input] Case {case_num}: DESTINATION not found – case skipped.")
            continue
        if source not in graph.nodes:
            print(
                f"[parse_input] Case {case_num}: SOURCE '{source}' is not a node in the graph – case skipped."
            )
            continue
        if destination not in graph.nodes:
            print(
                f"[parse_input] Case {case_num}: DESTINATION '{destination}' is not a node in the graph – case skipped."
            )
            continue

        cases.append({
            'case_num'   : case_num,
            'source'     : source,
            'destination': destination,
            'graph'      : graph,
        })

    if not cases:
        print("[parse_input] Warning: no valid cases found in input file.")

    return cases


print("Cell 2 loaded: parse_input() ready.")


Cell 2 loaded: parse_input() ready.


In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 │ IDA* Algorithm
# ─────────────────────────────────────────────────────────────────────────────
# IDA* (Iterative Deepening A*) combines the memory efficiency of DFS with
# the informed search of A*.
#
# Algorithm overview
# ──────────────────
# 1. Set initial f-cost threshold  T = h(start)
# 2. Run a depth-first search (DFS) where a branch is pruned as soon as
#    f(n) = g(n) + h(n) exceeds T.
# 3. If the goal is found within T → return the path (optimal).
# 4. Otherwise, update T to the minimum f-value that exceeded T, and repeat.
#
# Cost function
# ─────────────
#   g(n) = cumulative edge-weight cost from start to n along the current path.
#   h(n) = BFS_hops(n, goal) × min_edge_cost   (admissible heuristic)
#   f(n) = g(n) + h(n)                          (estimated total cost)
#
# Properties
# ──────────
#   Complete  : Yes (finds a solution if one exists on a finite graph)
#   Optimal   : Yes (because h is admissible)
#   Space     : O(b × d)  – only the current path is stored (vs O(b^d) for A*)
#   Time      : O(b^d)    – same asymptotic bound as A* in the worst case
# ─────────────────────────────────────────────────────────────────────────────


def ida_star(graph: Graph, start: str, goal: str):
    """
    Run IDA* on *graph* from *start* to *goal*.

    Parameters
    ----------
    graph : Graph   – populated Graph instance
    start : str     – source node name
    goal  : str     – destination node name

    Returns
    -------
    path           : list[str] – optimal path (start … goal), or None
    cost           : float     – total travel cost,           or float('inf')
    nodes_explored : int       – total search() calls (all iterations)
    """
    # ── Input validation ──────────────────────────────────────────────────────
    if start not in graph.nodes:
        print(f"[ida_star] Error: source node '{start}' not found in graph.")
        return None, float('inf'), 0
    if goal not in graph.nodes:
        print(f"[ida_star] Error: goal node '{goal}' not found in graph.")
        return None, float('inf'), 0

    # ── Pre-compute heuristic for every node ──────────────────────────────────
    h = graph.compute_heuristic(goal)

    if h[start] == float('inf'):
        print(f"[ida_star] Goal '{goal}' is not reachable from '{start}'.")
        return None, float('inf'), 0

    # ── Shared mutable counter (modified inside the nested function) ──────────
    nodes_explored = [0]   # list used so the nested function can mutate it

    # ── Recursive depth-limited search ───────────────────────────────────────
    def search(path: list, g: float, threshold: float):
        """
        Depth-first search pruned by f-cost threshold.

        Parameters
        ----------
        path      : current path as a stack (path[-1] is the current node)
        g         : cost accumulated from start to path[-1]
        threshold : current f-cost limit

        Returns
        -------
        (result, found_path, found_cost)
          result == -1          → goal reached; found_path and found_cost valid
          result == float('inf')→ branch exhausted with no successor
          result == f_value     → minimum f exceeding threshold in this subtree
        """
        node = path[-1]
        nodes_explored[0] += 1          # count every node visit
        f    = g + h.get(node, float('inf'))

        # ── Prune branch ──────────────────────────────────────────────────────
        if f > threshold:
            return f, None, 0.0

        # ── Goal check ────────────────────────────────────────────────────────
        if node == goal:
            return -1, list(path), g

        # ── Expand neighbours ─────────────────────────────────────────────────
        minimum = float('inf')
        for neighbour, edge_cost in graph.get_neighbors(node):
            if neighbour in path:
                continue                # avoid cycles on the current path

            path.append(neighbour)
            result, found_path, found_cost = search(path, g + edge_cost, threshold)
            path.pop()

            if result == -1:            # goal found – propagate upward
                return -1, found_path, found_cost

            if result < minimum:
                minimum = result        # track tightest next threshold

        return minimum, None, 0.0

    # ── Outer IDA* loop ───────────────────────────────────────────────────────
    threshold = h[start]              # initial bound = heuristic of start node
    path      = [start]

    while True:
        result, found_path, found_cost = search(path, 0.0, threshold)

        if result == -1:              # optimal path found
            return found_path, found_cost, nodes_explored[0]

        if result == float('inf'):    # goal is unreachable
            print(f"[ida_star] No path from '{start}' to '{goal}'.")
            return None, float('inf'), nodes_explored[0]

        threshold = result            # raise bound to next candidate f-value


print("Cell 3 loaded: ida_star() ready.")


Cell 3 loaded: ida_star() ready.


In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 │ Output Formatter + Main Execution
# ─────────────────────────────────────────────────────────────────────────────
# Reads  : inputPS1.txt  (parsed by Cell 2)
# Runs   : IDA* for each case (Cell 3)
# Writes : outputPS1.txt  AND  prints the same to stdout
# ─────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = 'inputPS1.txt'
OUTPUT_FILE = 'outputPS1.txt'


def format_cost(cost: float) -> str:
    """Return cost as an integer string if it is a whole number, else float."""
    return str(int(cost)) if cost == int(cost) else str(cost)


def format_case_result(case_num: int, source: str, destination: str,
                        path, cost: float, nodes_explored: int) -> str:
    """
    Build the output block for one case.

    Output format
    ─────────────
    === Case N ===
    Source      : <node>
    Destination : <node>
    Optimal Path: <node> -> <node> -> ...
    Total Travel Cost: <cost>
    Nodes Explored   : <count>
    Sequence of Locations: <node>, <node>, ...
    """
    lines = [f"=== Case {case_num} ==="]

    if path is None:
        lines += [
            f"Source      : {source}",
            f"Destination : {destination}",
            "Optimal Path: No path found",
            "Total Travel Cost: N/A",
            f"Nodes Explored   : {nodes_explored}",
            "Sequence of Locations: N/A",
        ]
    else:
        path_str     = " -> ".join(path)
        sequence_str = ", ".join(path)
        lines += [
            f"Source      : {source}",
            f"Destination : {destination}",
            f"Optimal Path: {path_str}",
            f"Total Travel Cost: {format_cost(cost)}",
            f"Nodes Explored   : {nodes_explored}",
            f"Sequence of Locations: {sequence_str}",
        ]

    return "\n".join(lines)


def write_output(blocks: list, output_file: str) -> None:
    """
    Write all formatted result blocks to *output_file* and print to stdout.

    Parameters
    ----------
    blocks      : list of formatted result strings (one per case)
    output_file : path to the output text file
    """
    full_output = "\n\n".join(blocks)

    with open(output_file, 'w') as fh:
        fh.write(full_output + "\n")

    print(full_output)
    print(f"\n[Output written to '{output_file}']")


# ── Main execution ────────────────────────────────────────────────────────────
def main(input_file: str = INPUT_FILE, output_file: str = OUTPUT_FILE) -> None:
    """
    End-to-end driver: parse → search → format → write.

    PEAS description of the Smart Waste Collection Robot Agent
    ──────────────────────────────────────────────────────────
    Performance : Minimise total travel cost (edge-weight sum on optimal path)
    Environment : Weighted road network (static, fully observable, deterministic)
    Actuators   : Move the robot between directly connected waste collection nodes
    Sensors     : Current location, edge weights, pre-built heuristic table
    """
    print(f"Reading input from '{input_file}' …\n")

    # ── Parse input ───────────────────────────────────────────────────────────
    try:
        cases = parse_input(input_file)
    except FileNotFoundError as exc:
        print(exc)
        return

    if not cases:
        print("No valid cases to process.")
        return

    # ── Run IDA* for each case ────────────────────────────────────────────────
    result_blocks = []

    for case in cases:
        n   = case['case_num']
        src = case['source']
        dst = case['destination']
        g   = case['graph']

        print(f"─── Case {n}: {src} → {dst} ───")

        path, cost, explored = ida_star(g, src, dst)
        block = format_case_result(n, src, dst, path, cost, explored)
        result_blocks.append(block)

    # ── Write consolidated output ─────────────────────────────────────────────
    print("\n" + "═" * 50)
    write_output(result_blocks, output_file)


# Run
main()


Reading input from 'inputPS1.txt' …

─── Case 1: MG_Road → Yelahanka ───
─── Case 2: A → E ───

══════════════════════════════════════════════════
=== Case 1 ===
Source      : MG_Road
Destination : Yelahanka
Optimal Path: MG_Road -> Electronic_City -> Whitefield -> Yelahanka
Total Travel Cost: 8
Nodes Explored   : 10
Sequence of Locations: MG_Road, Electronic_City, Whitefield, Yelahanka

=== Case 2 ===
Source      : A
Destination : E
Optimal Path: A -> C -> D -> E
Total Travel Cost: 5
Nodes Explored   : 22
Sequence of Locations: A, C, D, E

[Output written to 'outputPS1.txt']
